In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import fiona
import datetime
import shutil

In [ ]:
input_dir = Path.cwd().parent / "data" / "input_aanpassingen_db"
output_dir = Path.cwd().parent / "data" / "output_aanpassingen_db"
logbook_dir = Path.cwd().parent / "logbooks"
main_dir = Path.cwd().parent

In [ ]:
huidig = input_dir / "kansendatabase_huidige.gpkg"
nieuw = input_dir / "kansendatabase_nieuw.gpkg"

In [ ]:
# Check dat de lagen goed worden ingelezen
layers = fiona.listlayers(huidig)
assert set(layers) == set(
    [
        "primaire_keringen",
        "niet_primaire_keringen",
        "doorbraak_locaties_primair",
        "doorbraak_locaties_regionaal",
    ]
)

In [ ]:
# check the shape lengths of the new
# TODO:

In [ ]:
gdf_primair_kansen_huidig = gpd.read_file(huidig, layer="primaire_keringen")
gdf_niet_primair_huidig = gpd.read_file(huidig, layer="niet_primaire_keringen")
gdf_doorbraak_locaties_primair_huidig = gpd.read_file(
    huidig,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal_huidig = gpd.read_file(
    huidig,
    layer="doorbraak_locaties_regionaal",
)

In [ ]:
gdf_primair_kansen_nieuw = gpd.read_file(nieuw, layer="primaire_keringen")
gdf_niet_primair_nieuw = gpd.read_file(nieuw, layer="niet_primaire_keringen")
gdf_doorbraak_locaties_primair_nieuw = gpd.read_file(
    nieuw,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal_nieuw = gpd.read_file(
    nieuw,
    layer="doorbraak_locaties_regionaal",
)

In [ ]:
logbook = "# Loggboek verschillen kansendatabase\n\n"
logbook += f"## {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
logbook += f"huidige versie: {huidig.name} vs nieuwe verseie: {nieuw.name}\n\n"

Deze vergelijking is op basis van de kollom fid in qgis, dus geen aanpassingen maken aan die kolom

Geopandas leest deze standaard in

In [ ]:
# sanity checks on columns and dtypes for all 4 dataset pairs
datasets = [
    ("primair_kansen", gdf_primair_kansen_huidig, gdf_primair_kansen_nieuw),
    ("niet_primair", gdf_niet_primair_huidig, gdf_niet_primair_nieuw),
    (
        "doorbraak_locaties_primair",
        gdf_doorbraak_locaties_primair_huidig,
        gdf_doorbraak_locaties_primair_nieuw,
    ),
    (
        "doorbraak_locaties_regionaal",
        gdf_doorbraak_locaties_regionaal_huidig,
        gdf_doorbraak_locaties_regionaal_nieuw,
    ),
]

logbook += "### Check columns and dtypes\n\n"
for name, gdf_huidig, gdf_nieuw in datasets:
    if set(gdf_huidig.columns) != set(gdf_nieuw.columns):
        logbook += f"[{name}] Columns are different\n\n"
        verschil_huidig = set(gdf_huidig.columns) - set(gdf_nieuw.columns)
        verschil_nieuw = set(gdf_nieuw.columns) - set(gdf_huidig.columns)

        if len(verschil_huidig) > 0:
            logbook += f"alleen huidig: {verschil_huidig}\n\n"
        if len(verschil_nieuw) > 0:
            logbook += f"alleen nieuw: {verschil_nieuw}\n\n"

        # Alleen gedeelde kolommen vergelijken op dtype
        for col_name in set(gdf_huidig.columns) & set(gdf_nieuw.columns):
            if gdf_huidig[col_name].dtype != gdf_nieuw[col_name].dtype:
                logbook += (
                    f"[{name}] Column {col_name} has different dtypes:\n\n"
                    f"{gdf_huidig[col_name].dtype} vs {gdf_nieuw[col_name].dtype}\n\n"
                )

In [ ]:
# check nieuwe indices, handel
if logbook[-1] != "\n":
    logbook += "\n"
logbook += "### Check new rows\n\n"
for name, gdf_huidig, gdf_nieuw in datasets:
    if set(gdf_huidig.index) != set(gdf_nieuw.index):
        logbook += f"[{name}] Verschillende indices:\n\n"
        verschil_huidig = set(gdf_huidig.index) - set(gdf_nieuw.index)
        verschil_nieuw = set(gdf_nieuw.index) - set(gdf_huidig.index)
        if len(verschil_huidig) > 0:
            logbook += f"verwijderde rijen: {verschil_huidig}\n\n"
            verwijderde_rijen = gdf_huidig.loc[list(verschil_huidig)]
            for idx, row in verwijderde_rijen.iterrows():
                logbook += f"{row.T}\n"
        if len(verschil_nieuw) > 0:
            logbook += f"Nieuwe rijen: {verschil_nieuw}\n\n"
            nieuwe_rijen = gdf_nieuw.loc[list(verschil_nieuw)]
            for idx, row in nieuwe_rijen.iterrows():
                logbook += f"{row.T}\n"

In [ ]:
# check nieuwe indices, handel
logbook += "\n"
logbook += "### Check row differences\n\n"
lst_joined = []
for name, gdf_huidig, gdf_nieuw in datasets:
    # eerst alleen de inhoud
    joined_df: pd.DataFrame = gdf_huidig.join(
        gdf_nieuw, lsuffix="_huidig", rsuffix="_nieuw"
    )
    common_cols = gdf_huidig.columns.intersection(gdf_nieuw.columns)
    diff_mask = {}

    for c in common_cols:
        left = joined_df[f"{c}_huidig"]
        right = joined_df[f"{c}_nieuw"]

        if c == "geometry":  # geometry
            equal = left.geom_equals(right)
        else:
            equal = left.eq(right) | (left.isna() & right.isna())

        diff_mask[c] = ~equal

    diff_mask = pd.DataFrame(diff_mask, index=joined_df.index)
    changed_rows = diff_mask.any(axis=1)
    changed_cols = diff_mask.columns[diff_mask.any(axis=0)].tolist()

    if changed_rows.any():
        logbook += f"[{name}] {int(changed_rows.sum())} gewijzigde rijen; kolommen: {changed_cols}\n\n"
        cols_to_keep = [f"{c}_huidig" for c in changed_cols] + [
            f"{c}_nieuw" for c in changed_cols
        ]
        df_differences = joined_df.loc[changed_rows, cols_to_keep]
        lst_joined.append((name, df_differences))
        for idx, row in df_differences.iterrows():
            logbook += f"{row.T}\n"
    else:
        lst_joined.append((name, joined_df.iloc[0:0]))

In [ ]:
# Check op bressen:
logbook += "### Check bressen\n\n"
gdf_doorbraak_locaties_primair_nieuw

In [ ]:
# gdf_primair_kansen_nieuw['TRAJECT_ID']# .apply(lambda x: x.replace('#', '').split('-')[0])

In [ ]:
from IPython.display import Markdown, display

display(Markdown(logbook))

In [ ]:
# write output to file
nieuw_out = output_dir / "kansendatabase.gpkg"
shutil.copy(nieuw, nieuw_out)
nieuw_hoofd = main_dir / "kansendatabase.gpkg"
shutil.copy(nieuw, nieuw_hoofd)

with open(
    logbook_dir / f"logboek_{datetime.datetime.now().strftime('%Y_%m_%d')}.md", "w"
) as f:
    f.write(logbook)


driver = "GeoJSON"
gdf_primair_kansen_nieuw.to_file(
    output_dir / "kansendatabase_primaire_keringen.geojson",
    driver=driver,
)
gdf_niet_primair_nieuw.to_file(
    output_dir / "kansendatabase_niet_primaire_keringen.geojson",
    driver=driver,
)
gdf_doorbraak_locaties_primair_nieuw.to_file(
    output_dir / "kansendatabase_doorbraak_locaties_primair.geojson",
    driver=driver,
)
gdf_doorbraak_locaties_regionaal_nieuw.to_file(
    output_dir / "kansendatabase_doorbraak_locaties_regionaal.geojson",
    driver=driver,
)